# Stage 2 predictor data

This notebook calls the `gold_vol_pipeline.predictors` CLI to download the working `GC=F` gold-futures series and the selected daily predictor panel, align the observations to gold forecast origins, and save a prepared dataset plus run manifest. It writes raw and processed data beneath the Git-ignored `data/` directory.

FRED observations are downloaded as CSV directly from the public FRED website; no API key is required. Yahoo Finance data are also downloaded when the pipeline runs.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
if not (repo_root / 'pyproject.toml').is_file():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').is_file():
    raise FileNotFoundError('Could not locate the repository root (pyproject.toml).')

print(f'Repository: {repo_root}')

Repository: /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject


## Configure the download

The end date is exclusive, so the default avoids requesting today's potentially incomplete bar. Change the range when reproducing a specific analysis snapshot.

In [2]:
START_DATE = '2015-01-01'
END_DATE = pd.Timestamp.now().normalize().strftime('%Y-%m-%d')
HORIZON = 5
TARGET = 'realized_variance'

print(f'Download range: [{START_DATE}, {END_DATE})')
print(f'Target: {HORIZON}-session forward {TARGET}')

Download range: [2015-01-01, 2026-11-11)
Target: 5-session forward realized_variance


## Download and prepare the panel

The module obtains `GC=F`, DXY, S&P 500, and VIX observations from Yahoo Finance and Treasury yield, breakeven inflation, and WTI observations from FRED. FRED values are aligned using the project's one-business-day availability proxy, not historical release timestamps or vintages.

In [3]:
command = [
    sys.executable,
    '-m',
    'gold_vol_pipeline.predictors',
    '--start',
    START_DATE,
    '--end',
    END_DATE,
    '--horizon',
    str(HORIZON),
    '--target',
    TARGET,
]
kernel_env = os.environ.copy()
source_path = str(repo_root / 'src')
kernel_env['PYTHONPATH'] = os.pathsep.join(
    path for path in (source_path, kernel_env.get('PYTHONPATH', '')) if path
)
print('Running:', ' '.join(command[1:]))
result = subprocess.run(
    command,
    cwd=repo_root,
    check=True,
    capture_output=True,
    text=True,
    env=kernel_env,
)
print(result.stdout.strip())
if result.stderr.strip():
    print('Downloader messages:\n' + result.stderr.strip())

Running: -m gold_vol_pipeline.predictors --start 2015-01-01 --end 2026-11-11 --horizon 5 --target realized_variance


Wrote 2976 forecast rows to data/processed/stage2_multivariate.csv; run metadata saved to data/processed/stage2_multivariate_manifest.json.
Downloader messages:
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/Library/Frameworks/Python.frame

## Review the generated files

The processed panel retains each predictor's source observation date alongside its availability date. Missing values are not imputed; inspect coverage and freshness before fitting a multivariate model.

In [4]:
prepared_path = repo_root / 'data/processed/stage2_multivariate.csv'
manifest_path = repo_root / 'data/processed/stage2_multivariate_manifest.json'
prepared = pd.read_csv(prepared_path, parse_dates=['date'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

print(f'Prepared rows: {len(prepared):,}')
print(f'Forecast-origin range: {prepared["date"].min().date()} to {prepared["date"].max().date()}')
display(prepared.head())
display(pd.DataFrame(manifest['predictor_sources'].items(), columns=['column', 'source']))

Prepared rows: 2,976
Forecast-origin range: 2015-01-05 to 2026-11-03


,date,close,log_return,target_start_date,target_end_date,realized_variance,gold_volume,source,instrument,currency,...,sp500_log_return,sp500_log_return_observation_date,vix_close,vix_close_observation_date,treasury_10y_yield,treasury_10y_yield_observation_date,breakeven_inflation_10y,breakeven_inflation_10y_observation_date,wti_spot,wti_spot_observation_date
0,2015-01-05,1203.900024,0.014980,2015-01-06,2015-01-12,0.000442,470,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,...,-0.018447,2015-01-05,19.920000,2015-01-05,2.12,2015-01-02,1.71,2015-01-02,52.72,2015-01-02
1,2015-01-06,1219.300049,0.012711,2015-01-07,2015-01-13,0.000282,97,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,...,-0.008933,2015-01-06,21.120001,2015-01-06,2.04,2015-01-05,1.64,2015-01-05,50.05,2015-01-05
2,2015-01-07,1210.599976,-0.007161,2015-01-08,2015-01-14,0.000230,29,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,...,0.011563,2015-01-07,19.309999,2015-01-07,1.97,2015-01-06,1.56,2015-01-06,47.98,2015-01-06
3,2015-01-08,1208.400024,-0.001819,2015-01-09,2015-01-15,0.000815,92,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,...,0.017730,2015-01-08,17.010000,2015-01-08,1.96,2015-01-07,1.57,2015-01-07,48.69,2015-01-07
4,2015-01-09,1216.000000,0.006270,2015-01-12,2015-01-16,0.000868,43,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,...,-0.008439,2015-01-09,17.549999,2015-01-09,2.03,2015-01-08,1.62,2015-01-08,48.69,2015-01-07


,column,source
0,dxy_close,Yahoo Finance DX-Y.NYB (ICE U.S. Dollar Index)
1,sp500_close,Yahoo Finance ^GSPC (S&P 500 price index)
2,vix_close,Yahoo Finance ^VIX (CBOE Volatility Index)
3,sp500_log_return,Close-to-close log difference calculated from ...
4,treasury_10y_yield,FRED public CSV download DGS10 (10-year Treasu...
5,breakeven_inflation_10y,FRED public CSV download T10YIE (10-year break...
6,wti_spot,FRED public CSV download DCOILWTICO (WTI Cushi...
